# Final FLAN-T5 ACOS Model Training

This notebook prepares the final FLAN-T5 training pipeline for ACOS extraction. It is intentionally not executed here. Run the training cell only when the final retraining is approved.

In [3]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

c:\Users\Administrator\OneDrive\Desktop\Semester 2 2025-2026\FYP1\Code\course-acos-extraction.venv\Scripts\python.exe
2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 4070 SUPER


## 1. Project Overview

The project extracts Aspect-Category-Opinion-Sentiment (ACOS) quadruples from course evaluation feedback. The final model will fine-tune `google/flan-t5-base` using the standardized processed OATS-ABSA training data.

In [4]:
from pathlib import Path
import inspect
import json
import sys

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from postprocessing import parse_acos_output

CONFIG_PATH = PROJECT_ROOT / "config" / "final_training_config.json"

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = json.load(handle)

config

{'model_name': 'google/flan-t5-base',
 'train_file': 'data/processed/train.csv',
 'dev_file': 'data/processed/dev.csv',
 'test_file': 'data/processed/test.csv',
 'output_dir': 'models/final_flan_t5_acos_model',
 'prompt_template': 'Extract ACOS quadruples from this course review: {text}',
 'max_input_length': 256,
 'max_target_length': 128,
 'batch_size': 8,
 'learning_rate': 5e-05,
 'num_train_epochs': 8,
 'optimizer': 'AdamW'}

## 2. Dataset Loading

Load the already processed train, dev, and test CSV files. The test set is loaded only to confirm availability; it should be used after training for final evaluation.

In [5]:
def project_path(relative_path):
    path = Path(relative_path)
    return path if path.is_absolute() else PROJECT_ROOT / path

train_df = pd.read_csv(project_path(config["train_file"])).fillna("")
dev_df = pd.read_csv(project_path(config["dev_file"])).fillna("")
test_df = pd.read_csv(project_path(config["test_file"])).fillna("")

train_df.head()

,text,gold,domain,split
0,"I will strongly recommend "" Politics and Econo...",(implicit | course general | strongly recommen...,coursera,train
1,This course is also an asset for those advisin...,(course | course quality | implicit | positive),coursera,train
2,I have learned so much from this course though...,(course | material comprehensiveness | learned...,coursera,train
3,Yes it was challenging because it is good .,(implicit | course quality | challenging becau...,coursera,train
4,I can only say that I am glad I made the decis...,(course | course general | glad I made the dec...,coursera,train


## 3. Processed Dataset Statistics

These are the final confirmed counts for the standardized processed dataset.

In [6]:
def count_gold_tuples(frame):
    return int(frame["gold"].map(lambda value: len(parse_acos_output(value))).sum())


dataset_stats = pd.DataFrame([
    {"split": "train", "examples": len(train_df), "acos_tuples": count_gold_tuples(train_df)},
    {"split": "dev", "examples": len(dev_df), "acos_tuples": count_gold_tuples(dev_df)},
    {"split": "test", "examples": len(test_df), "acos_tuples": count_gold_tuples(test_df)},
])
dataset_stats

,split,examples,acos_tuples
0,train,16415,23204
1,dev,1980,2961
2,test,986,1444


## 4. Prompt and Target Format

Each review is converted into a prompt, and the model learns to generate the ACOS target string.

In [7]:
sample = train_df.iloc[0]
sample_prompt = config["prompt_template"].format(text=sample["text"])
sample_target = sample["gold"]

print("Prompt:")
print(sample_prompt)
print("\nGold target:")
print(sample_target)

Prompt:
Extract ACOS quadruples from this course review: I will strongly recommend " Politics and Economics of International Energy " to anyone working or thinking about working on the energy field .

Gold target:
(implicit | course general | strongly recommend | positive)


## 5. Model and Tokenizer Setup

Load the FLAN-T5 base tokenizer and model. This downloads or loads the base model only; it does not start training.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
model = AutoModelForSeq2SeqLM.from_pretrained(config["model_name"])

print(config["model_name"])
print("CUDA available:", torch.cuda.is_available())

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 10723.62it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


google/flan-t5-base
CUDA available: True


## 6. Training Configuration

Training uses AdamW through the Hugging Face Trainer. The output folder is protected by the script version, so existing model files are not overwritten accidentally.

In [9]:
ALLOW_OVERWRITE = False

training_config = {
    "model_name": config["model_name"],
    "batch_size": config["batch_size"],
    "learning_rate": config["learning_rate"],
    "num_train_epochs": config["num_train_epochs"],
    "optimizer": config["optimizer"],
    "max_input_length": config["max_input_length"],
    "max_target_length": config["max_target_length"],
    "output_dir": config["output_dir"],
    "allow_overwrite": ALLOW_OVERWRITE,
}
training_config

{'model_name': 'google/flan-t5-base',
 'batch_size': 8,
 'learning_rate': 5e-05,
 'num_train_epochs': 8,
 'optimizer': 'AdamW',
 'max_input_length': 256,
 'max_target_length': 128,
 'output_dir': 'models/final_flan_t5_acos_model',
 'allow_overwrite': False}

## 7. Training Procedure

The dataset class tokenizes the prompt as input and the ACOS string as the target label.

In [10]:
class AcosSeq2SeqDataset(Dataset):
    def __init__(self, frame, tokenizer, config):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.config = config

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        prompt = self.config["prompt_template"].format(text=str(row["text"]))
        target = str(row["gold"])

        model_inputs = self.tokenizer(
            prompt,
            max_length=int(self.config["max_input_length"]),
            truncation=True,
        )
        labels = self.tokenizer(
            text_target=target,
            max_length=int(self.config["max_target_length"]),
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return {key: torch.tensor(value) for key, value in model_inputs.items()}


train_dataset = AcosSeq2SeqDataset(train_df, tokenizer, config)
dev_dataset = AcosSeq2SeqDataset(dev_df, tokenizer, config)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)

## 8. Validation During Training

Validation uses the dev split at the end of each epoch. The test split is reserved for final evaluation only.

In [11]:
output_dir = project_path(config["output_dir"])

training_kwargs = {
    "output_dir": str(output_dir),
    "overwrite_output_dir": ALLOW_OVERWRITE,
    "do_train": True,
    "do_eval": True,
    "save_strategy": "epoch",
    "logging_strategy": "steps",
    "logging_steps": 50,
    "learning_rate": float(config["learning_rate"]),
    "per_device_train_batch_size": int(config["batch_size"]),
    "per_device_eval_batch_size": int(config["batch_size"]),
    "num_train_epochs": float(config["num_train_epochs"]),
    "weight_decay": 0.01,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "predict_with_generate": False,
    "optim": "adamw_torch",
    "report_to": "none",
    "fp16": torch.cuda.is_available(),
}
parameters = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
if "eval_strategy" in parameters:
    training_kwargs["eval_strategy"] = "epoch"
else:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer is prepared. Training has not started.")

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'overwrite_output_dir'

## 9. Saving the Final Model

The final model will be saved only after training is intentionally started and completed.

In [ ]:
def output_dir_is_nonempty(path):
    return path.exists() and any(path.iterdir())


if output_dir_is_nonempty(output_dir) and not ALLOW_OVERWRITE:
    raise RuntimeError(
        f"Output folder already exists and is not empty: {output_dir}. "
        "Set ALLOW_OVERWRITE = True only if you intentionally want to overwrite it."
    )

print(f"Output folder safety check passed: {output_dir}")

In [ ]:
# Run this cell only when ready to start final training
if output_dir_is_nonempty(output_dir) and not ALLOW_OVERWRITE:
    raise RuntimeError(
        f"Output folder already exists and is not empty: {output_dir}. "
        "Set ALLOW_OVERWRITE = True only if you intentionally want to overwrite it."
    )

trainer.train()
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))
print(f"Final model saved to: {output_dir}")

## 10. Final Evaluation Plan

After final training, generate predictions on `data/processed/test.csv`, evaluate exact-match Precision, Recall, and F1, and compare the final model against the previous deployed model.

## 11. Next Steps After Training

1. Generate final predictions from `models/final_flan_t5_acos_model/`.
2. Evaluate the final model on the all-domain test set.
3. Compare old deployed model metrics with final retrained model metrics.
4. Update README, report, and app only after real final metrics are available.